<a href="https://colab.research.google.com/github/1021114Carlos/MIT_MM_Finance/blob/Finance-shop/Courses/Derivative_Markets/M5_Black_Schole_Merton_and_Greeks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem Sets

In [1]:
from IPython.display import display, Math
import sympy as sp
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

Calculate value of European put option today

In [2]:
def euro_put_binomial_crr(s0, k, r, q, t, sigma, n):
  """
  European put price under crr binomial model
  """

  dt = t/n
  u = math.exp(sigma*math.sqrt(dt))
  d = 1.0/u
  R = math.exp((r - q)*dt)
  p = (R - d)/(u - d)
  disc = math.exp(-r*dt)

  # terminal stock price
  stock_prices = [s0*(u**j)*(d**(n - j)) for j in range(n + 1)]
  # put payoff
  values = [max(k - s, 0.0) for s in stock_prices]

  # backward induction
  for step in range(n, 0, -1):
    values = [disc*(p*values[j + 1] + (1.0 - p)*values[j])for j in range(step)]
  return values[0]



σ as a function of volatility

In [3]:
# Put price vs volatility

def price_vs_volatility(s0, k, r, q, t, n, sigmas):
  rows = []
  for sigma in sigmas:
    price = euro_put_binomial_crr(s0, k, r, q, t, sigma, n)
    rows.append({"sigma": sigma, "put_price": price})
  df = pd.DataFrame(rows)
  df["put_price"] = df["put_price"].round(6)
  return df

# usage
s0 = 375.0
k = 300.0
r = 0.01
q = 0.0
t = 0.5
n = 10
sigmas = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

df_vol = price_vs_volatility(s0, k, r, q, t, n, sigmas)
print(df_vol)



   sigma  put_price
0   0.20   1.086019
1   0.25   2.356337
2   0.30   5.319023
3   0.35   8.300596
4   0.40  11.291301
5   0.45  14.285612
6   0.50  17.311141


Price as a function of number of steps n and strike K

In [4]:
# put price relative to steps n and given strike k

def price_vs_steps(s0, k, r, q, t, sigma, n_list):
  rows = []
  for n in n_list:
    price = euro_put_binomial_crr(s0, k, r, q, t, sigma, n)
    rows.append({"n": n, "k": k, "put_price": price})
  df = pd.DataFrame(rows)
  df["put_price"] = df["put_price"].round(6)
  return df

# example to use

sigma = 0.25
n_list = list(range(10, 101, 10))

df300 = price_vs_steps(s0=375.0, k = 300.0, r=0.01, q=0.0, t=0.5, sigma=sigma, n_list=n_list)
df360 = price_vs_steps(s0=375.0, k = 360.0, r=0.01, q=0.0, t=0.5, sigma=sigma, n_list=n_list)

# print(f"k = 300 {df300} \n")
# print(f"k = 360 {df360}")

print(df300)
print("\n")
print(df360)



     n      k  put_price
0   10  300.0   2.356337
1   20  300.0   2.714365
2   30  300.0   2.777989
3   40  300.0   2.647561
4   50  300.0   2.763315
5   60  300.0   2.711934
6   70  300.0   2.740899
7   80  300.0   2.751653
8   90  300.0   2.699818
9  100  300.0   2.743575


     n      k  put_price
0   10  360.0  18.793586
1   20  360.0  18.590931
2   30  360.0  18.461655
3   40  360.0  18.373167
4   50  360.0  18.307922
5   60  360.0  18.257249
6   70  360.0  18.216401
7   80  360.0  18.223241
8   90  360.0  18.259663
9  100  360.0  18.283968


Greeks with 5% increase (numerical derivatives)

In [5]:
# Greeks with 5% increase

"""
returns a dict: delta, gamma, theta, rho, vega
theta is given in the usual option convention (negative of dp/dt)
"""
def greeks_5pct_bump(s0, k, r, q, t, sigma, n, bump=0.05):

  p0 = euro_put_binomial_crr(s0, k, r, q, t, sigma, n)

  # Delta and Gamma
  s_up = s0*(1 + bump)
  s_down = s0*(1 - bump)
  h_s = s0*bump

  p_up = euro_put_binomial_crr(s_up, k, r, q, t, sigma, n)
  p_down = euro_put_binomial_crr(s_down, k, r, q, t, sigma, n)

  # forward delta
  delta = (p_up - p0)/(s_up - s0)

  #symmetric gamma using two-sided deltas around s0
  delta_plus = (p_up - p0)/(h_s)
  delta_minus = (p0 - p_down)/h_s
  gamma = (delta_plus - delta_minus)/(s_up - s_down)

  # theta
  t_up = t*(1 + bump)
  p_t_up = euro_put_binomial_crr(s0, k, r, q, t_up, sigma, n)
  dp_dt = (p_t_up - p0)/(t_up - t)
  theta = -dp_dt  # as per convention


  # rho
  r_up = r*(1 + bump)
  p_r_up = euro_put_binomial_crr(s0, k, r_up, q, t, sigma, n)
  rho = (p_r_up - p0)/(r_up -r)

  # Vega

  sigma_up = sigma*(1 + bump)
  p_sigma_up = euro_put_binomial_crr(s0, k, r, q, t, sigma_up, n)
  vega = (p_sigma_up - p0)/(sigma_up -sigma)

  return {"Price": p0, "Delta": delta, "Gamma": gamma, "Theta": theta, "Rho": rho, "Vega": vega}

# Example use

greeks = greeks_5pct_bump(s0=375.0, k=300.0, r=0.01, q=0.0, t=0.5, sigma=0.25, n=10)
for name, value in greeks.items():
  print(f"{name:6s}: {value:.6f}")



Price : 2.356337
Delta : -0.043801
Gamma : 0.002653
Theta : -14.244658
Rho   : -14.952611
Vega  : 59.047721


2a)

In [ ]:
nodes = [{"name": "s0", "x": 0, "y": 105.000}, {"name": "su", "x": 1, "y": 115.581}, {"name": "sd", "x": 1, "y": 93.489},
         {"name": "suu", "x": 2, "y": 127.229}, {"name": "sud", "x": 2, "y": 102.910}, {"name": "sdd", "x": 2, "y": 83.240}]

edges_idx = [(0,1), (0, 2), (1, 3), (1, 4), (2, 4), (2, 5)]

edge_x = []
edge_y = []

for i, j in edges_idx:
  edge_x += [nodes[i]["x"], nodes[j]["x"], None]
  edge_y += [nodes[i]["y"], nodes[j]["y"], None]

edges_trace = go.Scatter(x=edge_x, y=edge_y, mode="lines", line=dict(width=1), hoverinfo="none", showlegend=False)

node_x = [n["x"] for n in nodes]
node_y = [n["y"] for n in nodes]

nodes_trace = go.Scatter(x=node_x, y=node_y, mode="markers+text", text = [f"{y:.3f}" for y in node_y], textposition='top center',
                         hovertext=[f"{n['name']}<br>t={n['x']}, s={n['y']:.3f}" for n in nodes], hoverinfo="text", marker=dict(size=10),
                         showlegend=False)


fig = go.Figure(data=[edges_trace, nodes_trace])

fig.update_layout(title="Binomial Stock Price Tree (h = 0.5 years)",
                  xaxis_title = "Time Step",
                  yaxis_title="Stock Price",
                  xaxis=dict(dtick=1, range=[-0.2, 2.2]),
                  template="simple_white")
fig.show()



In [6]:
print("Risk neutral probability \n")
display(Math(r"q* = \frac{{(1 + r)-d}}{{u - d}}"))
print("\n")
print("Risk neutral probability with continuous compounding.\n")
display(Math(r"q^* = \frac{e^{rt}-d}{u - d}"))

Risk neutral probability 



<IPython.core.display.Math object>



Risk neutral probability with continuous compounding.



<IPython.core.display.Math object>

In [7]:
# Find u and d
display(Math(r"u = \frac{{s_{u_1}}}{{s_0}}"))
print("\n")
display(Math(r"d = \frac{{s_{d_1}}}{{s_0}}"))


up = 115.581/105
down = 93.489/105
print("\n")
print(f"up = {up:.6f}, down = {down:.6f}")

<IPython.core.display.Math object>

<IPython.core.display.Math object>



up = 1.100771, down = 0.890371


b)

In [8]:
# Find volatility
display(Math(r"up = e^{(\sigma\sqrt{t})}"))

vol = math.log(up)/math.sqrt(0.5)

display(Math(rf"\sigma ={vol:.6f}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

c):

In [11]:
display(Math(r"q^* = \frac{e^{(r - \lambda)*t}-d}{u - d}"))

risk_neutra_prob = 0.4037
div_yield = r - (math.log(risk_neutra_prob*(up - down) + down)/t)
div_yield


<IPython.core.display.Math object>

0.060000007088832207